# Diabetes Prediction

Predicts diabetes from the Pima Indians Diabetes dataset (768 patients, 8 clinical features) with logistic regression and random forest.


In [1]:

import io, urllib.request
import pandas as pd
# Pima Indians Diabetes dataset (768 patients) - two mirrors, same data
urls = ["https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv",
        "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"]
d = None
for url in urls:
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        d = pd.read_csv(io.BytesIO(urllib.request.urlopen(req).read()))
        target = "Outcome" if "Outcome" in d.columns else "label"
        break
    except Exception as e:
        print("mirror failed:", e)
print(d.shape); d.head()


(768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
X = d.drop(columns=[target]); y = d[target].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
models = {"Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42)}
for name, m in models.items():
    m.fit(X_train, y_train); print(name, "accuracy:", round(m.score(X_test, y_test), 4))
best_name = max(models, key=lambda n: models[n].score(X_test, y_test))
best_model = models[best_name]
print("Best:", best_name)


Logistic Regression accuracy: 0.7344


Random Forest accuracy: 0.7552
Best: Random Forest


In [3]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pred = best_model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"{best_name} accuracy: {acc:.4f}")
print(classification_report(y_test, pred))
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i,j] > cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=120); plt.show()


Random Forest accuracy: 0.7552
              precision    recall  f1-score   support

           0       0.79      0.85      0.82       125
           1       0.67      0.58      0.62        67

    accuracy                           0.76       192
   macro avg       0.73      0.72      0.72       192
weighted avg       0.75      0.76      0.75       192



## Result

See README.md for the verified accuracy. Diabetes screening models support — never replace — medical advice.
